# Notebook 05 — NDVI Statistics and Exploratory Classes
**Phase:** calculate valid-pixel statistics and five NDVI intervals used in the completed project.

Intervals: below 0; 0 to <0.15; 0.15 to <0.30; 0.30 to <0.45; and 0.45 or above. These are descriptive NDVI bins, not validated land-cover classes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
NDVI_PATH = Path(r"data/processed/Three_Mandals_NDVI_Clipped_20261009.tif")
OUT_DIR = Path("reports")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Raster exists:", NDVI_PATH.exists())

## 1. Read the raster and calculate statistics

In [ ]:
if NDVI_PATH.exists():
    with rasterio.open(NDVI_PATH) as src:
        masked = src.read(1, masked=True).astype("float32")
        values = np.asarray(masked.filled(np.nan), dtype="float32")
        valid = np.isfinite(values)
        if src.nodata is not None: valid &= values != src.nodata
        pixel_area_km2 = abs(src.res[0] * src.res[1]) / 1_000_000
        print("Dimensions:", src.width, "x", src.height, "CRS:", src.crs,
              "resolution:", src.res, "NoData:", src.nodata)
    v = values[valid]
    if v.size:
        print("Valid pixels:", v.size, "Min:", v.min(), "Median:", np.median(v),
              "Mean:", v.mean(), "Max:", v.max(), "Std:", v.std(),
              "Valid area km²:", v.size * pixel_area_km2)
else:
    print("Set NDVI_PATH to the existing final clipped raster.")

## 2. Calculate class counts and areas

In [ ]:
if NDVI_PATH.exists() and valid.any():
    edges = [-np.inf, 0.0, 0.15, 0.30, 0.45, np.inf]
    labels = ["Below 0", "0 to <0.15", "0.15 to <0.30", "0.30 to <0.45", "0.45 and above"]
    class_id = np.digitize(values, edges[1:-1], right=False) + 1
    class_id[~valid] = 0
    total = int(valid.sum())
    rows = []
    for cid, label in enumerate(labels, 1):
        count = int(((class_id == cid) & valid).sum())
        rows.append({"class_id":cid,"ndvi_interval":label,"pixels":count,
                     "percent_valid_pixels":100*count/total if total else 0,
                     "area_km2":count*pixel_area_km2})
    class_stats = pd.DataFrame(rows)
    display(class_stats)
    class_stats.to_csv(OUT_DIR / "ndvi_class_area_statistics.csv", index=False)
else:
    print("Skipped — raster not loaded.")

In [ ]:
if NDVI_PATH.exists() and valid.any():
    plt.figure(figsize=(9,4.5))
    plt.hist(values[valid], bins=80)
    plt.xlabel("NDVI"); plt.ylabel("Pixel count")
    plt.title("Three-mandal NDVI distribution — 9 October 2026")
    plt.tight_layout(); plt.show()
    plt.figure(figsize=(9,4.5))
    plt.bar(class_stats["ndvi_interval"], class_stats["percent_valid_pixels"])
    plt.ylabel("Percent of valid pixels"); plt.xticks(rotation=25, ha="right")
    plt.title("Exploratory NDVI interval distribution")
    plt.tight_layout(); plt.show()

## 3. Completed-project reference values
Previously reported overall values: minimum −0.2507, maximum 0.6997, mean 0.2786, median 0.2634, standard deviation 0.1259; valid-data coverage 93.07%; valid area 505.889 km².

| Interval | Share | Area (km²) |
|---|---:|---:|
| Below 0 | 1.16% | 5.8901 |
| 0 to <0.15 | 12.81% | 64.7955 |
| 0.15 to <0.30 | 47.59% | 240.7611 |
| 0.30 to <0.45 | 27.51% | 139.1638 |
| 0.45 and above | 10.93% | 55.2785 |

Treat these as reference values from the completed workflow; recompute from your actual files before publication.

## Troubleshooting and interpretation
- Percentages may differ slightly from 100% due to rounding.
- Unexpected values outside [-1, 1] require checking NoData, zero denominators, scaling, and input bands.
- Area calculation assumes projected pixel dimensions in metres.
- Do not equate NDVI bins with confirmed crop/urban/forest classes.
- One date cannot establish seasonal change, yield, or plant health.

## Phase outcome
Keep small CSVs and PNG plots in GitHub. Keep oversized raster outputs locally and document how to regenerate them.